In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Target Table Path (Silver Order items)

In [0]:
target_table = "ecommerce.silver.slv_order_items"

### Read Bronze Table

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_order_items")

### Incremental Watermarking (Filter new records using _ingested_at)
##### (Process New Data Only)

In [0]:
if spark.catalog.tableExists(target_table):
    max_silver_time = spark.table(target_table).select(F.max("processed_time")).collect()[0][0]
    if max_silver_time:
        df_bronze = df_bronze.filter(F.col("_ingested_at") > max_silver_time)

### Clean and Transform Data

In [0]:
# 3. Clean and Transform Data
if df_bronze.count() > 0:

    df_clean = (df_bronze
        # Quantity transformation
        .withColumn(
            "quantity",
            F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
        )
        # Unit price cleaning
        .withColumn(
            "unit_price",
            F.regexp_replace("unit_price", r"[\$]", "").cast("double")
        )
        # Discount percentage cleaning
        .withColumn(
            "discount_pct",
            F.regexp_replace("discount_pct", "%", "").cast("double")
        )
        # Coupon code processing
        .withColumn(
            "coupon_code", F.lower(F.trim(F.col("coupon_code")))
        )
        # Channel processing
        .withColumn(
            "channel",
            F.when(F.col("channel") == "web", "Website")
            .when(F.col("channel") == "app", "Mobile")
            .otherwise(F.col("channel"))
        )
        # Date & Timestamp casting
        .withColumn("dt", F.to_date("dt", "yyyy-MM-dd"))
        .withColumn(
            "order_ts",
            F.coalesce(
                F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),
                F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")
            )
        )
        # Item sequence casting
        .withColumn("item_seq", F.col("item_seq").cast("int"))
        # Tax amount cleaning
        .withColumn(
            "tax_amount",
            F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double")
        )
        # Audit timestamp
        .withColumn("processed_time", F.current_timestamp())
        # Deduplication AFTER casting integer keys
        .dropDuplicates(["order_id", "item_seq"])
    )

    # 4. Smart Write Logic (First Time Run vs Subsequent Runs)
    if not spark.catalog.tableExists(target_table):
        # FIRST TIME RUN: Create table and save initial batch
        (df_clean.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
        print("Silver Fact Table created successfully for the first time!")

    else:
        # SUBSEQUENT RUNS: Incremental MERGE (Upsert) using Composite Key
        silver_delta_table = DeltaTable.forName(spark, target_table)

        (silver_delta_table.alias("target")
            .merge(
                df_clean.alias("source"),
                "target.order_id = source.order_id AND target.item_seq = source.item_seq"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("New batch merged successfully into the Silver Fact Table!")

else:
    print("No new records found in the Bronze Table to process.")

Silver Fact Table created successfully for the first time!


### Check Number of Rows 

In [0]:
# %sql
# SELECT COUNT(*) AS Number_of_rows FROM ecommerce.bronze.brz_order_items
# UNION ALL
# SELECT COUNT(*) AS Number_of_rows FROM ecommerce.silver.slv_order_items;

Number_of_rows
183378
183378


### Check Number of Rows and Columns of order_items Table in each Layer
##### Before and After ingest into Silver

In [0]:
# %sql
# SELECT 'Bronze' AS table_layer, COUNT(*) AS total_rows,
#     (SELECT COUNT(*) FROM system.information_schema.columns WHERE table_catalog = 'ecommerce' AND table_schem = 'bronze' AND table_name = 'brz_order_items') AS total_columns
# FROM ecommerce.bronze.brz_order_items

# UNION ALL

# SELECT 'Silver' AS table_layer, COUNT(*) AS total_rows,
#     (SELECT COUNT(*) FROM system.information_schema.columns WHERE table_catalog = 'ecommerce' AND table_schema = 'silver' AND table_name = 'slv_order_items') AS total_columns
# FROM ecommerce.silver.slv_order_items;

table_layer,total_rows,total_columns
Bronze,183378,16
Silver,183378,17


In [0]:
%sql

-- DROP TABLE ecommerce.silver.slv_order_items;